In [1]:
import os
import sys
import pandas as pd
import numpy as np
import yaml

pd.set_option("display.max_columns", None)

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(os.path.join(PROJECT_ROOT, "src"))

with open(os.path.join(PROJECT_ROOT, "config", "config.yaml")) as f:
    config = yaml.safe_load(f)

SEED = config["seed"]
RAW_DIR = os.path.join(PROJECT_ROOT, config["paths"]["raw_dir"])
INTERIM_DIR = os.path.join(PROJECT_ROOT, config["paths"]["interim_dir"])
os.makedirs(INTERIM_DIR, exist_ok=True)

In [2]:
pk = pd.read_csv(os.path.join(RAW_DIR, config["raw_files"]["pakistan_raw"]))
pk["y_depression"] = (pk["Labelling"] == "Depressed").astype(int)

print("Final label distribution:")
print(pk["y_depression"].value_counts())
print(pk["y_depression"].value_counts(normalize=True).round(4))

# --- Data cleaning: fix whitespace-duplicated categories + 2 data-entry typos ---
# Found via the SHAP bar charts in Phase 9: several categorical columns had
# trailing-whitespace duplicates (e.g. "Yes" vs "Yes ") that were silently
# treated as different categories by one-hot encoding since Phase 4.
text_cat_cols = ["Female Education", "Husband Education", "Working Status", "Previous Miscarriage",
                  "Sufficient Money for Basic Needs", "Current Appereance Acceptance",
                  "Male Gender Preference", "Gravida"]

for col in text_cat_cols:
    pk[col] = pk[col].astype(str).str.strip()
    pk[col] = pk[col].replace("nan", pd.NA)  # restore true missing values

pk["Current Appereance Acceptance"] = pk["Current Appereance Acceptance"].replace("YY", "Yes")
pk["Gravida"] = pk["Gravida"].replace("Multigravida+D1191", "Multigravida")

print("\nCleaned category sets (verify each column shows no duplicates):")
for col in text_cat_cols:
    print(f"  {col}: {sorted(pk[col].dropna().unique().tolist())}")

pk.to_csv(os.path.join(INTERIM_DIR, "pakistan_interim.csv"), index=False)
print(f"\nSaved: {os.path.join(INTERIM_DIR, 'pakistan_interim.csv')}")

Final label distribution:
y_depression
1    9078
0    4930
Name: count, dtype: int64
y_depression
1    0.6481
0    0.3519
Name: proportion, dtype: float64

Cleaned category sets (verify each column shows no duplicates):
  Female Education: ['Graduation', 'Intermediate', 'Matric', 'Middle', 'Primary', 'Uneducated']
  Husband Education: ['Graduation', 'Intermediate', 'Matric', 'Middle', 'Primary', 'Uneducated']
  Working Status: ['Housewife', 'Working Lady']
  Previous Miscarriage: ['No', 'Yes']
  Sufficient Money for Basic Needs: ['No', 'Yes']
  Current Appereance Acceptance: ['No', 'Yes']
  Male Gender Preference: ['No', 'Yes']
  Gravida: ['Multigravida', 'Primigravida']

Saved: C:\maternal-mental-health-risk\data/interim\pakistan_interim.csv


In [3]:
ar = pd.read_excel(os.path.join(RAW_DIR, config["raw_files"]["arab5_data"]))

dep_items = [f"DepressionRec{i}" for i in range(1, 9)] + ["Depressionec9"]
anx_items = [f"Anxietyrec{i}" for i in range(1, 8)]

ar["dep_items_complete"] = ar[dep_items].notna().all(axis=1)
ar["anx_items_complete"] = ar[anx_items].notna().all(axis=1)
ar["y_depression"] = (ar["Depression_Rec1"] >= 2).astype(int)   # Moderate+Severe = 1
ar["y_anxiety"] = (ar["Anxeity_Rec"] >= 2).astype(int)

print("Depression: complete-item rows vs default-imputed rows")
print(ar["dep_items_complete"].value_counts())
print("\nAnxiety: complete-item rows vs default-imputed rows")
print(ar["anx_items_complete"].value_counts())

Depression: complete-item rows vs default-imputed rows
dep_items_complete
False    2070
True     1499
Name: count, dtype: int64

Anxiety: complete-item rows vs default-imputed rows
anx_items_complete
False    2623
True      946
Name: count, dtype: int64


In [4]:
print("DECISION (documented for the manuscript's Methods section):")
print("Primary analysis for Arab5 uses only rows with complete item-level data")
print("(dep_items_complete == True / anx_items_complete == True), avoiding the")
print("country-linked default-imputation bias identified in notebook 02, Cell 10.")
print("Full-n models (including default-imputed rows) are run separately as a")
print("documented sensitivity analysis, not the primary result.")
print()
print(f"Primary depression analytic sample: n = {ar['dep_items_complete'].sum()}")
print(f"Primary anxiety analytic sample: n = {ar['anx_items_complete'].sum()}")

ar.to_csv(os.path.join(INTERIM_DIR, "arab5_interim.csv"), index=False)
print(f"\nSaved: {os.path.join(INTERIM_DIR, 'arab5_interim.csv')}")
print("(Both flags and both labels are kept in this one file — notebook 04 will")
print("filter to the primary subset by default and use the full n for sensitivity runs.)")

DECISION (documented for the manuscript's Methods section):
Primary analysis for Arab5 uses only rows with complete item-level data
(dep_items_complete == True / anx_items_complete == True), avoiding the
country-linked default-imputation bias identified in notebook 02, Cell 10.
Full-n models (including default-imputed rows) are run separately as a
documented sensitivity analysis, not the primary result.

Primary depression analytic sample: n = 1499
Primary anxiety analytic sample: n = 946

Saved: C:\maternal-mental-health-risk\data/interim\arab5_interim.csv
(Both flags and both labels are kept in this one file — notebook 04 will
filter to the primary subset by default and use the full n for sensitivity runs.)


In [5]:
ug = pd.read_csv(os.path.join(RAW_DIR, config["raw_files"]["uganda"]))
ug["age_numeric"] = pd.to_numeric(ug["age"], errors="coerce")

print(ug["Anxiety"].describe())
print(ug["Anxiety"].quantile([0.5, 0.6, 0.7, 0.75, 0.8]))

count    14325.000000
mean         3.642653
std          1.611571
min          0.000000
25%          3.000000
50%          3.000000
75%          5.000000
max          9.000000
Name: Anxiety, dtype: float64
0.50    3.0
0.60    4.0
0.70    4.0
0.75    5.0
0.80    5.0
Name: Anxiety, dtype: float64


In [6]:
# No externally validated clinical cutoff exists for this instrument (unlike PHQ-9/GAD-7).
# Anxiety >= 4 sits close to the sample median/60th percentile and yields a reasonably
# balanced binary split -- documented here as an investigator-defined threshold, same
# category of limitation as the reference manuscript's own composite risk_label.
CUTOFF = 4
ug["y_anxiety_binary"] = (ug["Anxiety"] >= CUTOFF).astype(int)

print(f"Binary label at cutoff >= {CUTOFF}:")
print(ug["y_anxiety_binary"].value_counts())
print(ug["y_anxiety_binary"].value_counts(normalize=True).round(4))
print("\nBoth the continuous Total Score/Anxiety AND this binary label are kept --")
print("primary analysis for Uganda can run as regression, with this binary version")
print("as a secondary classification framing for comparability with the other datasets.")

ug.to_csv(os.path.join(INTERIM_DIR, "uganda_interim.csv"), index=False)
print(f"\nSaved: {os.path.join(INTERIM_DIR, 'uganda_interim.csv')}")

Binary label at cutoff >= 4:
y_anxiety_binary
0    7303
1    7022
Name: count, dtype: int64
y_anxiety_binary
0    0.5098
1    0.4902
Name: proportion, dtype: float64

Both the continuous Total Score/Anxiety AND this binary label are kept --
primary analysis for Uganda can run as regression, with this binary version
as a secondary classification framing for comparability with the other datasets.

Saved: C:\maternal-mental-health-risk\data/interim\uganda_interim.csv


In [7]:
pn = pd.read_csv(os.path.join(RAW_DIR, config["raw_files"]["postnatal"]))

scale_map = {
    "No": 0, "Not at all": 0,
    "Sometimes": 1, "Maybe": 1, "Not interested to say": 1,
    "Yes": 2, "Often": 2, "Two or more days a week": 2,
}

all_items = ["Feeling sad or Tearful", "Irritable towards baby & partner",
             "Trouble sleeping at night", "Problems concentrating or making decision",
             "Overeating or loss of appetite", "Feeling anxious", "Feeling of guilt",
             "Problems of bonding with baby", "Suicide attempt"]

for col in all_items:
    pn[col + "_enc"] = pn[col].map(scale_map)

print(pn[[c + "_enc" for c in all_items]].describe())

       Feeling sad or Tearful_enc  Irritable towards baby & partner_enc  \
count                 1503.000000                           1497.000000   
mean                     1.007984                              1.037408   
std                      0.840037                              0.838538   
min                      0.000000                              0.000000   
25%                      0.000000                              0.000000   
50%                      1.000000                              1.000000   
75%                      2.000000                              2.000000   
max                      2.000000                              2.000000   

       Trouble sleeping at night_enc  \
count                    1503.000000   
mean                        1.443779   
std                         0.896434   
min                         0.000000   
25%                         0.000000   
50%                         2.000000   
75%                         2.000000   
max 

In [8]:
label_items = ["Feeling sad or Tearful", "Feeling anxious", "Feeling of guilt", "Suicide attempt"]
predictor_items = ["Irritable towards baby & partner", "Trouble sleeping at night",
                    "Problems concentrating or making decision", "Overeating or loss of appetite",
                    "Problems of bonding with baby"]

assert set(label_items).isdisjoint(predictor_items), "Overlap would recreate the same leakage issue as Phase 2"
print("Label-construction items (used ONLY to build the proxy label):")
print(label_items)
print("\nPredictor items (used ONLY as model features, plus Age):")
print(predictor_items)

Label-construction items (used ONLY to build the proxy label):
['Feeling sad or Tearful', 'Feeling anxious', 'Feeling of guilt', 'Suicide attempt']

Predictor items (used ONLY as model features, plus Age):
['Irritable towards baby & partner', 'Trouble sleeping at night', 'Problems concentrating or making decision', 'Overeating or loss of appetite', 'Problems of bonding with baby']


In [9]:
pn["proxy_score"] = pn[[c + "_enc" for c in label_items]].sum(axis=1)
print(pn["proxy_score"].describe())
print(pn["proxy_score"].value_counts().sort_index())

CUTOFF = 5   # out of a possible 0-8, using 4 items scored 0-2 each
base_label = (pn["proxy_score"] >= CUTOFF)
suicide_override = (pn["Suicide attempt"] == "Yes")
rescued = (suicide_override & ~base_label).sum()

pn["y_ppd_risk"] = (base_label | suicide_override).astype(int)

print(f"\nBase label positive (score >= {CUTOFF}): {base_label.sum()} ({base_label.mean():.4f})")
print(f"Additional rows flagged High Risk via suicide-attempt override: {rescued}")
print(f"Final label positive rate: {pn['y_ppd_risk'].mean():.4f}")

count    1503.000000
mean        3.952096
std         1.357661
min         0.000000
25%         3.000000
50%         4.000000
75%         5.000000
max         8.000000
Name: proxy_score, dtype: float64
proxy_score
0.0     13
1.0      8
2.0    233
3.0    231
4.0    549
5.0    300
6.0    125
7.0     25
8.0     19
Name: count, dtype: int64

Base label positive (score >= 5): 469 (0.3120)
Additional rows flagged High Risk via suicide-attempt override: 240
Final label positive rate: 0.4717


In [10]:
pn.to_csv(os.path.join(INTERIM_DIR, "postnatal_interim.csv"), index=False)
print(f"Saved: {os.path.join(INTERIM_DIR, 'postnatal_interim.csv')}")

print("\nLIMITATION TO CARRY INTO THE MANUSCRIPT (Discussion/Limitations section):")
print("y_ppd_risk is an investigator-defined proxy label, not a validated clinical")
print("instrument score (unlike PHQ-9/GAD-7 in the other three datasets). Results on")
print("this dataset should be reported as a supplementary/exploratory robustness check,")
print("not a primary finding -- consistent with the framing set in the project roadmap.")

Saved: C:\maternal-mental-health-risk\data/interim\postnatal_interim.csv

LIMITATION TO CARRY INTO THE MANUSCRIPT (Discussion/Limitations section):
y_ppd_risk is an investigator-defined proxy label, not a validated clinical
instrument score (unlike PHQ-9/GAD-7 in the other three datasets). Results on
this dataset should be reported as a supplementary/exploratory robustness check,
not a primary finding -- consistent with the framing set in the project roadmap.
